# Experiment 2.2c notebook: toy study of initial momentum-imbalance compression under layerwise rescaling

**Pair identity.** This notebook is the presentation / analysis counterpart to:
`experiments/Tier_2_Symmetry_Reparametrization_Tests/Exp_2.2_Layerwise_Rescaling_Robustness/2.2c_Self_Correction_Timescale/run_experiment.py`

**Purpose of this first completion pass.** The notebook now **imports and runs the script's experiment function directly** instead of re-implementing the training loop. This keeps the notebook and script numerically aligned while making the notebook substantially more explicit about scope, metrics, runtime, caveats, and interpretation.

## Truthful scope framing

This remains a **toy deep-linear Muon-style study**. The current metric is not a full self-correction / equilibration timescale. What is measured is:

1. the per-step momentum-imbalance ratio
   $\max_\ell \|m_\ell\|_F / \min_\ell \|m_\ell\|_F$,
2. a **50%-compression half-life**: first recorded imbalance index where the ratio drops below half of its first recorded value,
3. a **training onset** index: first recorded pre-update loss index where loss falls below `0.99 × initial loss`.

The present notebook therefore supports claims about **early imbalance compression** and **loss-onset delay** under rescaling. It does **not** by itself establish a full self-correction timescale, an RG fixed-point gap, or general asymptotic scale-invariant behavior.


## Reproducibility and notebook execution policy

- The notebook searches upward from the current working directory to find the repository root and the paired `run_experiment.py` file.
- All experiment numerics come from `run_experiment.run_experiment()`.
- The notebook is responsible only for reproducibility logging, tables, figures, and calibrated interpretation.
- The script also returns execution provenance (timestamp, Python executable/version, NumPy version, cwd, and history-collection flag), which is logged below.
- The core default setup is intentionally preserved from the original 2.2c pair: 4 layers, width 32, 500 steps, 5 seeds, and `c ∈ {1, 10, 100, 1000, 10000}`.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import display
except Exception:
    display = None

EXPERIMENT_RELATIVE_DIR = Path(
    'experiments/Tier_2_Symmetry_Reparametrization_Tests/'
    'Exp_2.2_Layerwise_Rescaling_Robustness/'
    '2.2c_Self_Correction_Timescale'
)
SCRIPT_PATH_REL = EXPERIMENT_RELATIVE_DIR / 'run_experiment.py'


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / SCRIPT_PATH_REL).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate repository root containing ' + str(SCRIPT_PATH_REL)
    )


REPO_ROOT = find_repo_root(Path.cwd())
EXPERIMENT_DIR = REPO_ROOT / EXPERIMENT_RELATIVE_DIR
SCRIPT_PATH = REPO_ROOT / SCRIPT_PATH_REL

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.spec_from_file_location('exp_2_2c_run_experiment', SCRIPT_PATH)
exp22c = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp22c)

print(f'Repository root: {REPO_ROOT}')
print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Imported script: {SCRIPT_PATH}')


In [ ]:
def fmt_float(value, fmt='.3g'):
    try:
        value = float(value)
    except Exception:
        return str(value)
    return format(value, fmt) if np.isfinite(value) else 'nan'


def fmt_pm(mean_value, std_value, mean_fmt='.1f', std_fmt='.1f'):
    return f"{fmt_float(mean_value, mean_fmt)} ± {fmt_float(std_value, std_fmt)}"


def pad_histories(histories):
    max_len = max((len(hist) for hist in histories), default=0)
    arr = np.full((len(histories), max_len), np.nan)
    for idx, hist in enumerate(histories):
        arr[idx, :len(hist)] = hist
    return arr


def show_table(rows, title=None):
    if title:
        print(title)
    if not rows:
        print('(no rows)')
        return
    if pd is not None and display is not None:
        display(pd.DataFrame(rows))
        return
    columns = list(rows[0].keys())
    widths = {}
    for col in columns:
        widths[col] = max(len(str(col)), max(len(str(row.get(col, ''))) for row in rows))
    header = ' | '.join(str(col).ljust(widths[col]) for col in columns)
    divider = '-+-'.join('-' * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))


## Run the paired script and log reproducibility details

The next cell executes the script's `run_experiment()` function once, then exposes its structured results for notebook presentation. This is the single numerical source of truth for the notebook.


In [ ]:
notebook_start = time.perf_counter()
results = exp22c.run_experiment()
notebook_call_runtime = time.perf_counter() - notebook_start

config = results['config']
c_values = config['c_values']
seed_schedule = results['seed_schedule']
summary_by_c = {entry['c']: entry for entry in results['aggregate_summary']}
per_c = results['per_c']
hypothesis_tests = results['hypothesis_tests']
metric_notes = results['metric_notes']
provenance = results.get('provenance', {})

print(f"Experiment ID: {results['experiment_id']}")
print(f"Title: {results['title']}")
print(f"Script path: {results['script_path']}")
print(f"Script runtime reported by run_experiment(): {results['runtime_seconds']:.2f} s")
print(f"Notebook wall-clock for the function call: {notebook_call_runtime:.2f} s")
print()
print('Execution provenance:')
for key in ['generated_at_utc', 'python_executable', 'python_version', 'platform', 'numpy_version', 'cwd_at_invocation', 'collect_histories']:
    if key in provenance:
        print(f"  {key}: {provenance[key]}")
print()
print('Configuration:')
for key in ['dim', 'n_layers', 'num_steps', 'lr', 'momentum_beta', 'ns_iters', 'num_seeds', 'batch_size', 'c_values']:
    print(f"  {key}: {config[key]}")
print()
print('Seed schedule:')
for seed_info in seed_schedule:
    print(
        f"  seed_idx={seed_info['seed_idx']} | data_seed={seed_info['data_seed']} | "
        f"weight_seed={seed_info['weight_seed']}"
    )
print()
print('Metric notes / caveats carried directly from the script:')
for key, note in metric_notes.items():
    print(f"  - {key}: {note}")


## Aggregate half-life / onset summary table

The table below summarizes the current default sweep. The key operational caveat is repeated here:

- `loss_history[0]` is a **pre-update** measurement,
- `imbalance_history[0]` is a **post-first-update** measurement.

So `training onset` and `50%-compression half-life` are useful descriptive diagnostics, but their raw step indices should not be over-interpreted as sharing an identical time origin.


In [ ]:
summary_rows = []
for c in c_values:
    summary = summary_by_c[c]
    summary_rows.append({
        'c': c,
        'mean initial grad imbalance': fmt_float(summary['mean_initial_gradient_imbalance'], '.2e'),
        'mean first momentum imbalance': fmt_float(summary['mean_first_recorded_momentum_imbalance'], '.2e'),
        'mean final momentum imbalance': fmt_float(summary['mean_final_momentum_imbalance'], '.2e'),
        'half-life (mean ± std)': fmt_pm(summary['mean_half_life'], summary['std_half_life'], '.1f', '.1f'),
        'onset (mean ± std)': fmt_pm(summary['mean_training_onset'], summary['std_training_onset'], '.1f', '.1f'),
        'finite half-life seeds': f"{summary['finite_half_life_count']}/{summary['num_seeds']}",
        'raw onset < half-life seeds': f"{summary['raw_index_onset_before_half_life_count']}/{summary['raw_index_comparison_count']}",
    })

show_table(summary_rows, title='Aggregate summary by rescaling factor c')


## Imbalance trajectories by rescaling factor

Each panel shows the recorded momentum-imbalance ratio over optimization steps for one `c` value.

- thin colored traces: individual seeds
- thick black trace: across-seed mean
- dashed red line: mean 50%-compression half-life when finite

The y-axis is logarithmic because the initial imbalance spans many orders of magnitude.


In [ ]:
fig, axes = plt.subplots(len(c_values), 1, figsize=(10, 2.35 * len(c_values)), sharex=True, constrained_layout=True)
if len(c_values) == 1:
    axes = [axes]

for ax, c in zip(axes, c_values):
    seed_results = per_c[c]['seed_results']
    histories = [seed_result['imbalance_history'] for seed_result in seed_results]
    padded = pad_histories(histories)
    mean_history = np.nanmean(padded, axis=0) if padded.size else np.array([])

    for seed_result in seed_results:
        hist = seed_result['imbalance_history']
        ax.plot(np.arange(len(hist)), hist, alpha=0.25, linewidth=1.2)

    if mean_history.size:
        ax.plot(np.arange(len(mean_history)), mean_history, color='black', linewidth=2.2, label='seed mean')

    mean_half_life = summary_by_c[c]['mean_half_life']
    if np.isfinite(mean_half_life):
        ax.axvline(mean_half_life, color='tab:red', linestyle='--', linewidth=1.4, label='mean half-life')

    ax.set_yscale('log')
    ax.set_ylabel(f'c={c}\nimbalance')
    ax.grid(True, alpha=0.3)

axes[0].legend(loc='upper right', ncol=2)
axes[-1].set_xlabel('Recorded imbalance index (post-update clock)')
fig.suptitle('Momentum-imbalance trajectories under layerwise rescaling', y=1.02)
plt.show()


### Interpretation of the imbalance plot

What is supported:

- Larger `c` produces dramatically larger initial momentum imbalance.
- For `c ≥ 100`, the **50%-compression** event happens almost immediately under the current metric.
- Despite that rapid early compression, the final recorded imbalance can still remain far above 1, especially for the largest `c` values.

What is **not** supported by this figure alone:

- that the system has fully rebalanced,
- that there is a single well-estimated global self-correction timescale,
- that an RG/fixed-point theory has been empirically established by this toy diagnostic.


## Loss trajectories and training-onset delay

These panels show the corresponding loss histories for the same sweep.

- thin colored traces: individual seeds
- thick black trace: across-seed mean
- dashed green line: mean training-onset index when finite

The x-axis now follows the **loss clock**, which is recorded before each optimizer update. This is why onset-vs-half-life comparisons should be treated carefully.


In [ ]:
fig, axes = plt.subplots(len(c_values), 1, figsize=(10, 2.35 * len(c_values)), sharex=True, constrained_layout=True)
if len(c_values) == 1:
    axes = [axes]

for ax, c in zip(axes, c_values):
    seed_results = per_c[c]['seed_results']
    histories = [seed_result['loss_history'] for seed_result in seed_results]
    padded = pad_histories(histories)
    mean_history = np.nanmean(padded, axis=0) if padded.size else np.array([])

    for seed_result in seed_results:
        hist = seed_result['loss_history']
        ax.plot(np.arange(len(hist)), hist, alpha=0.25, linewidth=1.2)

    if mean_history.size:
        ax.plot(np.arange(len(mean_history)), mean_history, color='black', linewidth=2.2, label='seed mean')

    mean_onset = summary_by_c[c]['mean_training_onset']
    if np.isfinite(mean_onset):
        ax.axvline(mean_onset, color='tab:green', linestyle='--', linewidth=1.4, label='mean onset')

    ax.set_yscale('log')
    ax.set_ylabel(f'c={c}\nloss')
    ax.grid(True, alpha=0.3)

axes[0].legend(loc='upper right', ncol=2)
axes[-1].set_xlabel('Recorded loss index (pre-update clock)')
fig.suptitle('Loss trajectories and delayed onset at large rescaling', y=1.02)
plt.show()


### Interpretation of the loss plot

What is supported:

- Mild rescaling (`c=10`, `c=100`) still allows very early loss reduction.
- Stronger rescaling (`c=1000`, `c=10000`) delays loss decrease substantially.
- Therefore, **fast 50% imbalance compression does not imply immediate learning**.

This matters because it narrows the original overclaim. The present pair can show that Muon-style updates often compress imbalance quickly, but it cannot honestly claim that learning generally begins *before* rebalancing has meaningfully completed.


## Half-life vs onset summary view

This compact figure compares the mean half-life and mean onset across `c` on the same vertical axis. Error bars show the across-seed standard deviation.

Again, the two curves live on slightly different operational clocks, so the overlay is descriptive rather than a clock-synchronized causal comparison.


In [ ]:
log_c_values = np.log10(np.array(c_values, dtype=float))
half_means = np.array([summary_by_c[c]['mean_half_life'] for c in c_values], dtype=float)
half_stds = np.array([summary_by_c[c]['std_half_life'] for c in c_values], dtype=float)
onset_means = np.array([summary_by_c[c]['mean_training_onset'] for c in c_values], dtype=float)
onset_stds = np.array([summary_by_c[c]['std_training_onset'] for c in c_values], dtype=float)

fig, ax = plt.subplots(figsize=(8.5, 5.0))
mask_half = np.isfinite(half_means)
mask_onset = np.isfinite(onset_means)

ax.errorbar(
    log_c_values[mask_half],
    half_means[mask_half],
    yerr=half_stds[mask_half],
    fmt='o-',
    linewidth=2,
    capsize=4,
    label='50% imbalance-compression half-life',
)
ax.errorbar(
    log_c_values[mask_onset],
    onset_means[mask_onset],
    yerr=onset_stds[mask_onset],
    fmt='s-',
    linewidth=2,
    capsize=4,
    label='loss onset',
)

ax.set_xticks(log_c_values)
ax.set_xticklabels([str(c) for c in c_values])
ax.set_xlabel('Rescaling factor c')
ax.set_ylabel('Recorded step index')
ax.set_title('Summary of mean half-life and mean onset versus rescaling')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


### Interpretation of the summary view

The main empirical picture from this first pass is:

- initial imbalance grows explosively with `c`,
- the current **50%-compression** metric saturates very early for large `c`,
- training onset does **not** track that saturation: it becomes much later at the largest `c`.

So the current pair supports **rapid initial compression but not full, uniformly useful rebalancing**.


## Current T1 / T2 / T3 outputs, now reported with caveats

The script still returns the pair's historical T1/T2/T3 outputs so future threads can track continuity, but the notebook now reports them in a more calibrated way.


In [ ]:
t1 = hypothesis_tests['T1']
t2 = hypothesis_tests['T2']
t3 = hypothesis_tests['T3']

print('T1: half-life versus log10(c)')
print(f"  question: {t1['question']}")
print(f"  slope (steps/decade): {fmt_float(t1['slope_steps_per_decade'], '.3f')}")
print(f"  intercept: {fmt_float(t1['intercept'], '.3f')}")
print(f"  R^2: {fmt_float(t1['r_squared'], '.3f')}")
print(f"  legacy bound pass (<50 steps/decade): {t1['legacy_bound_pass']}")
print(f"  stronger monotone-log-growth support (0 <= slope < 50): {t1['monotone_log_growth_support']}")
print(f"  caveat: {t1['caveat']}")
print()

show_table(
    [
        {
            'c': entry['c'],
            'mean half-life': fmt_float(entry['mean_half_life'], '.1f'),
            'finite?': entry['finite'],
            '< 50?': entry['under_50'],
        }
        for entry in t2['per_c']
    ],
    title='T2 control-range readout (c <= 1000)'
)
print(f"legacy finite-only pass: {t2['legacy_finite_only_pass']}")
print(f"strict all-reported pass: {t2['strict_all_reported_pass']}")
print(f"nonfinite c values: {t2['nonfinite_c_values']}")
print(f"caveat: {t2['caveat']}")
print()

show_table(
    [
        {
            'c': entry['c'],
            'mean onset': fmt_float(entry['mean_training_onset'], '.1f'),
            'mean half-life': fmt_float(entry['mean_half_life'], '.1f'),
            'raw onset < half-life?': entry['raw_index_onset_before_half_life'],
            'comparable?': entry['comparable'],
        }
        for entry in t3['per_c']
    ],
    title='T3 per-c readout under the current raw index convention'
)
print(f"legacy any-pass: {t3['legacy_any_pass']}")
print(f"all comparable c pass: {t3['all_comparable_c_pass']}")
print(f"support fraction across comparable c values: {fmt_float(t3['support_fraction'], '.2f')}")
print(f"caveat: {t3['caveat']}")


## Calibrated conclusion for the present pair

This first implementation pass makes the script and notebook consistent and substantially more careful.

### What the pair now shows

- Under this toy deep-linear setup, larger layerwise rescaling creates very large initial cross-layer momentum imbalance.
- Muon-style updates often compress that imbalance **very quickly at the 50% level**.
- However, large `c` can still leave large residual imbalance and significantly delay loss reduction.

### What the pair does **not** show yet

- a full self-correction / equilibration timescale,
- synchronized evidence that learning generally begins before meaningful rebalancing is complete,
- direct confirmation of RG fixed-point or spectral-gap claims.

### Practical readout

The pair is now suitable as a **serious first-pass notebook + script baseline** for this toy study. A stronger future pass would add an additional non-saturating imbalance metric if the project still wants to defend the phrase **self-correction timescale** more literally.
